# Kev Data Collection on Grummon et al. (2025) FOP Nutrition Label RCT (Default: CPU)

In [1]:
# Step 1: Load Grummon et al. (2025) RCT dataset and subsample the 200 participants (1,200 choice tasks, seed=42)
import importlib
from pathlib import Path
import grummon_ols
import kev_eval

importlib.reload(kev_eval)
importlib.reload(grummon_ols)
from grummon_ols import load_grummon_rct, subsample_grummon_participants
from kev_eval import (
    build_kev_prompt_items,
    collect_kev_prompts,
    free_kev_memory,
    load_kev_model,
    preview_kev_item,
)

DEVICE = "cpu"
N_LLM_PARTICIPANTS = 200
SUBSAMPLE_SEED = 42
DATA_DIR = Path("data")

raw_df = load_grummon_rct()
sub_df = subsample_grummon_participants(raw_df, n_participants=N_LLM_PARTICIPANTS, seed=SUBSAMPLE_SEED)
print(f"Subsampled {sub_df['CASEID'].nunique()} participants ({len(sub_df):,} tasks) across {sub_df['arm'].nunique()} arms")

Subsampled 200 participants (1,200 tasks) across 6 arms


In [2]:
# Step 2: Build Kev prompt items for Random Persona (empty state) and Respondent Persona (10 shopper attributes, no Country)
kev_items_random = build_kev_prompt_items(sub_df, persona_mode="random", include_country=False)
kev_items_persona = build_kev_prompt_items(sub_df, persona_mode="persona", include_country=False)

print("=== Sample Random Persona Kev Prompt ===")
print(preview_kev_item(kev_items_random[0]))
print("\n=== Sample Respondent Persona Kev Prompt ===")
print(preview_kev_item(kev_items_persona[0]))

=== Sample Random Persona Kev Prompt ===


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

70 tokens; state 1 tokens (no truncation, strict=True)
<|fim_prefix|>
<|fim_middle|>
Click on the product you think is healthier, or click 'They're the same' if you think the two products are equally healthy.
<|box_start|>Ocean Spray 100% Cranberry juice: Front-of-package label: None<|box_end|>
<|box_start|>Ocean Spray Cranberry Juice Cocktail: Front-of-package label: None<|box_end|>
<|box_start|>They're the same<|box_end|>
<|fim_suffix|>

=== Sample Respondent Persona Kev Prompt ===


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

166 tokens; state 97 tokens (no truncation, strict=True)
<|fim_prefix|>
Sex: Female
Age: 30-49
Education: College graduate/some postgrad
Income: $50,000 - $75,000
Race: White
Hispanic or Latino: No
SNAP / Food Stamps recipient: No
Nutrition Facts label usage: Most of the time
Share of groceries bought online: Less than 25%
Nutrition literacy score: 4 out of 4
<|fim_middle|>
Click on the product you think is healthier, or click 'They're the same' if you think the two products are equally healthy.
<|box_start|>Ocean Spray 100% Cranberry juice: Front-of-package label: None<|box_end|>
<|box_start|>Ocean Spray Cranberry Juice Cocktail: Front-of-package label: None<|box_end|>
<|box_start|>They're the same<|box_end|>
<|fim_suffix|>


In [4]:
# Step 3: Collect Kev 0.8B Base & Fine-Tuned predictions on CPU (set N_PERSONAS=None for all 200 participants)
N_PERSONAS_08B = None

model = None
free_kev_memory()
model = load_kev_model("jaredpalmer/kev-0.8b", device=DEVICE, temperature=None)

recs_08b_base_random = collect_kev_prompts(
    model,
    kev_items_random,
    DATA_DIR / "kev-0.8b-base_random.jsonl",
    model_name="jaredpalmer/kev-0.8b",
    n_personas=N_PERSONAS_08B,
    temperature=None,
)
recs_08b_base_persona = collect_kev_prompts(
    model,
    kev_items_persona,
    DATA_DIR / "kev-0.8b-base_persona.jsonl",
    model_name="jaredpalmer/kev-0.8b",
    n_personas=N_PERSONAS_08B,
    temperature=None,
)

model = None
free_kev_memory()
FT_08B_CKPT = "../../runs/kev-0.8b-resp-issp+twin2k-grouped/best"
model = load_kev_model(FT_08B_CKPT, device=DEVICE, temperature=None)

recs_08b_ft_random = collect_kev_prompts(
    model,
    kev_items_random,
    DATA_DIR / "kev-0.8b-resp-issp+twin2k-grouped_best_random.jsonl",
    model_name="kev-0.8b-resp-issp+twin2k-grouped/best",
    n_personas=N_PERSONAS_08B,
    temperature=None,
)
recs_08b_ft_persona = collect_kev_prompts(
    model,
    kev_items_persona,
    DATA_DIR / "kev-0.8b-resp-issp+twin2k-grouped_best_persona.jsonl",
    model_name="kev-0.8b-resp-issp+twin2k-grouped/best",
    n_personas=N_PERSONAS_08B,
    temperature=None,
)

model = None
free_kev_memory()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[kev-0.8b-base_random.jsonl] Complete: all 1,200/1,200 requested items already saved (total in file: 1,200).
[kev-0.8b-base_persona.jsonl] Complete: all 1,200/1,200 requested items already saved (total in file: 1,200).


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[kev-0.8b-resp-issp+twin2k-grouped_best_random.jsonl] Complete: all 1,200/1,200 requested items already saved (total in file: 1,200).
[kev-0.8b-resp-issp+twin2k-grouped_best_persona.jsonl] Collecting 822 missing items across 137 unique (state, arm) groups on device=cpu (model='kev-0.8b-resp-issp+twin2k-grouped/best', T=1.0000)...


kev-0.8b-resp-issp+twin2k-grouped_best_persona (cpu):   0%|          | 0/137 [00:00<?, ?it/s]

In [ ]:
# Step 4: Collect Kev 9B Base & Fine-Tuned predictions on CPU (Random Persona uses only 6 arm groups for all 200 participants)
N_PERSONAS_9B = None

model = None
free_kev_memory()
model = load_kev_model("jaredpalmer/kev-9b", device=DEVICE, temperature=None)

recs_9b_base_random = collect_kev_prompts(
    model,
    kev_items_random,
    DATA_DIR / "kev-9b-base_random.jsonl",
    model_name="jaredpalmer/kev-9b",
    n_personas=200,
    temperature=None,
)
recs_9b_base_persona = collect_kev_prompts(
    model,
    kev_items_persona,
    DATA_DIR / "kev-9b-base_persona.jsonl",
    model_name="jaredpalmer/kev-9b",
    n_personas=N_PERSONAS_9B,
    temperature=None,
)

model = None
free_kev_memory()
FT_9B_CKPT = "../../runs/kev-9b-resp-issp+twin2k-grouped-newvalid/best"
model = load_kev_model(FT_9B_CKPT, device=DEVICE, temperature=None)

recs_9b_ft_random = collect_kev_prompts(
    model,
    kev_items_random,
    DATA_DIR / "kev-9b-resp-issp+twin2k-grouped-newvalid_best_random.jsonl",
    model_name="kev-9b-resp-issp+twin2k-grouped-newvalid/best",
    n_personas=200,
    temperature=None,
)
recs_9b_ft_persona = collect_kev_prompts(
    model,
    kev_items_persona,
    DATA_DIR / "kev-9b-resp-issp+twin2k-grouped-newvalid_best_persona.jsonl",
    model_name="kev-9b-resp-issp+twin2k-grouped-newvalid/best",
    n_personas=N_PERSONAS_9B,
    temperature=None,
)

model = None
free_kev_memory()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

[kev-9b-base_random.jsonl] Complete: all 1,200/1,200 requested items already saved (total in file: 1,200).
[kev-9b-base_persona.jsonl] Collecting 1,128 missing items across 188 unique (state, arm) groups on device=cpu (model='jaredpalmer/kev-9b', T=2.1936)...


... (3 outputs left out)

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

[kev-9b-resp-issp+twin2k-grouped-newvalid_best_random.jsonl] Complete: all 1,200/1,200 requested items already saved (total in file: 1,200).
[kev-9b-resp-issp+twin2k-grouped-newvalid_best_persona.jsonl] Collecting 1,128 missing items across 188 unique (state, arm) groups on device=cpu (model='kev-9b-resp-issp+twin2k-grouped-newvalid/best', T=1.0000)...


kev-9b-resp-issp+twin2k-grouped-newvalid_best_persona (cpu):   0%|          | 0/188 [00:00<?, ?it/s]

: 